# Keyword Spacing Attack on Spam Emails

This notebook performs an evasion attack by inserting spaces into common spam keywords. 
This can bypass simple keyword-based filters or classifiers while still being legible to the human eye.

## Import Required Libraries

We use Pandas for data manipulation, re (regular expressions) for text processing, and randomization for spacing variation.

In [2]:
import pandas as pd
import re
import random
from collections import Counter

## Step 1: Load Dataset

We load the SMS spam test dataset, which contains labeled spam and ham messages.

In [142]:
# Step 1: Load Enron dataset
df = pd.read_csv("../dataset/sms/test.csv")

## Step 2: Define Known Short Spam Keywords

A manually defined list of short spam-related words (e.g., "win", "buy", "hot") that are often targeted by filters.

In [143]:
# Step 2: Define 100 common short spam words manually
short_spam_words = [
    "win", "buy", "sex", "fun", "hot", "off", "now", "get", "low", "new",
    "act", "big", "top", "pro", "try", "out", "yes", "see", "pay", "ads",
    "job", "map", "fix", "xan", "cpa", "age", "url", "all", "wow", "add",
    "biz", "web", "ask", "app", "run", "pop", "hit", "zip", "via", "ufo",
    "opt", "net", "rep", "tax", "fly", "tip", "god", "men", "max",
    "qik", "you", "her", "him", "our", "its", "rx", "med",
    "kit", "sum", "inc", "edu", "cam", "log", "atm",
    "lot", "usd", "eur", "fee", "mlm", "seo", "eon", "vps", "vpn",
    "api", "aim", "tag", "per", "two", "sky", "boy", "gal",
    "diy", "roi", "vip", "spy", "pen", "bio", "sms", "free", "cash", 
    "deal", "earn", "gift", "loan", "save", "sale",
    "cheap", "offer", "bonus", "click", "risk", "trial", "code", "join",
    "fast", "easy", "best", "only", "more", "extra", "plus", "bucks",
    "prize", "score", "hot", "rate", "fees", "rich", "gold", "fund",
    "sell", "shop", "ads", "spam", "credit", "money", "income",
    "profit", "penny", "dollar", "promo", "freebie", "cashback", "earnings"
]

## Step 3: Extract Longer Spam Words Heuristically

We extract frequently occurring longer words from spam messages using basic frequency filtering.

In [144]:
# Step 3: Extract common long spam keywords from spam emails

# Define a function to extract frequently occurring long words from a list of texts
def extract_heuristic_spam_candidates(texts, min_freq=5, min_len=4, top_n=1000):
    words = []
    # Process each spam email
    for text in texts:
        # Convert to lowercase and replace non-word characters with spaces
        text = re.sub(r'\W+', ' ', text.lower())
        # Split text into words and add them to the list
        words += text.split()
    # Count frequency of each word across all spam emails
    common = Counter(words)
    # Return a list of words that appear at least `min_freq` times and are at least `min_len` characters long
    # Limit the result to the top_n most frequent words
    return [word for word, freq in common.items() if freq >= min_freq and len(word) >= min_len][:top_n]

spam_emails = df[df["target"] == "spam"]["email"]
long_spam_words = extract_heuristic_spam_candidates(spam_emails)

In [145]:
spam_emails = df[df["target"] == "spam"]["email"]
long_spam_words = extract_heuristic_spam_candidates(spam_emails)

In [146]:
len(long_spam_words)

88

## Step 4: Combine and Filter Keyword List

Merge short and long keywords into one list and remove undesired terms (e.g., common non-spam words like “subject”).

In [147]:
# Step 4: Combine and clean keyword list
spam_keywords = list(set(long_spam_words + short_spam_words))
spam_keywords = [w for w in spam_keywords if w.lower() != "subject"]

## Step 5: Define Spacing Rules

Short words get fully spaced (e.g., "win" → "w i n"), while long words may be partially or fully spaced to simulate random obfuscation.

In [148]:
# Step 5: Define spacing logic

# Function to apply spacing to a given word
def spaced(word):
    # If the word is very short, insert a space between all letters (e.g., "win" → "w i n")
    if len(word) < 4:
        return " ".join(list(word))  # Full spacing for short words

    # Randomly choose between fully spacing the word or partially inserting a space
    mode = random.choice(["full", "partial"])

    if mode == "full":
        # Insert spaces between all characters (e.g., "offer" → "o f f e r")
        return " ".join(list(word))
    else:
        # Insert a single space at a random position (not at the beginning or end)
        idx = random.randint(1, len(word) - 2)
        return word[:idx] + " " + word[idx:]

# Function to apply spacing attack on a given text
def apply_spacing_attack(text, spam_words):
    # Iterate through each known spam-related word
    for word in spam_words:
        if len(word) < 2:
            continue  # Skip very short words to avoid false positives

        # Create a case-insensitive regex pattern to match the word as a whole word
        pattern = re.compile(rf'\b{re.escape(word)}\b', re.IGNORECASE)

        # Replace the word in the text with a spaced version using the `spaced` function
        text = pattern.sub(lambda m: spaced(m.group(0)), text)

    # Return the modified text with spacing obfuscations
    return text


## Step 6: Apply Attack to Spam Messages

We apply the spacing attack **only to messages labeled as spam** by scanning each message and spacing spam-related keywords.

In [149]:
# Step 6: Apply the attack to spam messages only
df["email_spaced"] = df["email"]
spam_indices = df[df["target"] == "spam"].index

In [150]:
for idx in spam_indices:
    df.at[idx, "email_spaced"] = apply_spacing_attack(df.at[idx, "email"], spam_keywords)

## Preview Spaced Messages

Display a few modified messages to inspect how spacing was applied.

In [151]:
df.head(22)

,email,target,email_spaced
0,"Oh right, ok. I'll make sure that i do loads o...",ham,"Oh right, ok. I'll make sure that i do loads o..."
1,I am in tirupur. call you da.,ham,I am in tirupur. call you da.
2,No that just means you have a fat head,ham,No that just means you have a fat head
3,"You have won ?1,000 cash or a ?2,000 prize! To...",spam,"Y o u h ave won ?1,000 c a s h or a ?2,000 pr ..."
4,Come aftr &lt;DECIMAL&gt; ..now i m cleaning t...,ham,Come aftr &lt;DECIMAL&gt; ..now i m cleaning t...
5,Friendship poem: Dear O Dear U R Not Near But ...,ham,Friendship poem: Dear O Dear U R Not Near But ...
6,Wot about on wed nite I am 3 then but only til 9!,ham,Wot about on wed nite I am 3 then but only til 9!
7,Dont talk to him ever ok its my word.,ham,Dont talk to him ever ok its my word.
8,Congrats kano..whr s the treat maga?,ham,Congrats kano..whr s the treat maga?
9,Eh u remember how 2 spell his name... Yes i di...,ham,Eh u remember how 2 spell his name... Yes i di...


## Step 7: Save Modified Dataset

We export the updated dataset with spacing modifications for further use.

In [152]:
# Step 7: Save the new file
df.to_csv("../dataset/sms/spacing/test_spacing.csv", index=False)